# 🔴 Hard: Conv2D Forward (NumPy)

Implement a 2-D convolution forward pass over `NCHW` tensors, using only NumPy.

### Core Idea

A convolution is a linear layer with two constraints bolted on: **local connectivity** (each output
sees only a $K \times K$ patch) and **weight sharing** (the same kernel slides over every position).
That is what buys translation equivariance and cuts parameters from $O(HW \cdot H'W')$ to $O(K^2)$
per channel pair.

$$\text{out}[n, f, i, j] = b_f + \sum_{c}\sum_{u=0}^{K_H-1}\sum_{v=0}^{K_W-1} x[n,\, c,\, is+u-p,\, js+v-p]\; w[f,\, c,\, u,\, v]$$

Read the shapes as a contract: `x` is `(N, C, H, W)`, `w` is `(F, C, KH, KW)`, `b` is `(F,)`. Each
output channel $f$ is one filter that spans **all** input channels — a convolution is only spatially
local, never channel-local.

$$H_\text{out} = \left\lfloor \frac{H + 2p - K_H}{s} \right\rfloor + 1$$

Three things worth internalising:

- **It is cross-correlation, not convolution.** No kernel flipping. The math community's flip does not
  survive into deep learning because the kernel is learned — a flipped kernel is just as learnable.
- **Padding preserves resolution.** `p = (K-1)/2` with `s = 1` keeps `H_out == H`, which is why odd
  kernels (3, 5, 7) dominate.
- **The loop you write is not the loop that runs.** Real implementations use *im2col*: flatten every
  patch into a row, and the whole convolution becomes one big GEMM. Loop over the *output positions*
  (there are few) and vectorise over batch, channels and filters — never loop over N or C.

### Signature
```python
def conv2d_forward(x, w, b, stride=1, padding=0):
    # x: (N, C, H, W)   w: (F, C, KH, KW)   b: (F,)
    # returns: (N, F, H_out, W_out)
    ...
```

### Rules
- Pure **NumPy** — no PyTorch, no `scipy.signal`
- Zero padding on the spatial dims only
- Cross-correlation (do **not** flip the kernel)

### Example
```
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(4, 3, 3, 3)
b = np.random.randn(4)
conv2d_forward(x, w, b, stride=1, padding=0).shape  ->  (2, 4, 6, 6)
conv2d_forward(x, w, b, stride=2, padding=1).shape  ->  (2, 4, 4, 4)
```

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def conv2d_forward(x, w, b, stride=1, padding=0):
    # x: (N, C, H, W);  w: (F, C, KH, KW);  b: (F,)
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(4, 3, 3, 3)
b = np.random.randn(4)

print("stride=1 padding=0:", conv2d_forward(x, w, b, 1, 0).shape, "(expect (2, 4, 6, 6))")
print("stride=1 padding=1:", conv2d_forward(x, w, b, 1, 1).shape, "(expect (2, 4, 8, 8) — resolution preserved)")
print("stride=2 padding=1:", conv2d_forward(x, w, b, 2, 1).shape, "(expect (2, 4, 4, 4))")

# A center-tap kernel must reproduce the input
ident = np.zeros((1, 1, 3, 3)); ident[0, 0, 1, 1] = 1.0
one = np.random.randn(1, 1, 5, 5)
print("identity kernel   :", np.allclose(conv2d_forward(one, ident, np.zeros(1), 1, 1), one))

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("numpy_conv2d")